<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/07_temas_actuales/70_aprendizaje_por_refuerzo.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Aprendizaje por refuerzo: Q-learning

**Pregunta guía:** ¿Cómo aprende un agente de consecuencias demoradas?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Proceso de decisión de Markov

Un MDP contiene estados $s$, acciones $a$, transición, recompensa $r$ y
descuento $\gamma$. La función óptima satisface

$$Q^*(s,a)=E\left[r+\gamma\max_{a'}Q^*(s',a')\right].$$

Q-learning aproxima esa ecuación con
$Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$.
Es *off-policy*: el objetivo usa la acción codiciosa aunque la conducta
explore con $\varepsilon$-greedy.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

FILAS,COLUMNAS=6,6
INICIO=(5,0); META=(0,5); OBSTÁCULOS={(1,1),(1,2),(2,2),(3,2),(4,4)}
ACCIONES=[(-1,0),(1,0),(0,-1),(0,1)]
FLECHAS=np.array(["↑","↓","←","→"])

def paso(estado,acción):
    dr,dc=ACCIONES[acción]; candidato=(estado[0]+dr,estado[1]+dc)
    fuera=not(0<=candidato[0]<FILAS and 0<=candidato[1]<COLUMNAS)
    siguiente=estado if fuera or candidato in OBSTÁCULOS else candidato
    recompensa=10.0 if siguiente==META else -1.0
    return siguiente,recompensa,siguiente==META

def índice(estado): return estado[0]*COLUMNAS+estado[1]


In [ ]:
SEMILLA=42; rng=np.random.default_rng(SEMILLA)
Q=np.zeros((FILAS*COLUMNAS,len(ACCIONES))); retornos=[]
alpha=.15; gamma=.97
for episodio in range(2500):
    estado=INICIO; retorno=0; epsilon=max(.03,1-episodio/1800)
    for _ in range(250):
        if rng.random()<epsilon: acción=rng.integers(len(ACCIONES))
        else: acción=np.argmax(Q[índice(estado)])
        siguiente,r,fin=paso(estado,acción)
        objetivo=r if fin else r+gamma*np.max(Q[índice(siguiente)])
        Q[índice(estado),acción]+=alpha*(objetivo-Q[índice(estado),acción])
        estado=siguiente; retorno+=r
        if fin: break
    retornos.append(retorno)

media=np.convolve(retornos,np.ones(100)/100,mode="valid")
plt.plot(media); plt.xlabel("episodio"); plt.ylabel("retorno medio (100)"); plt.show()


In [ ]:
política=np.full((FILAS,COLUMNAS)," ",dtype=object)
valor=np.max(Q,axis=1).reshape(FILAS,COLUMNAS)
for f in range(FILAS):
    for c in range(COLUMNAS):
        if (f,c) in OBSTÁCULOS: política[f,c]="■"
        elif (f,c)==META: política[f,c]="★"
        else: política[f,c]=FLECHAS[np.argmax(Q[índice((f,c))])]
print(política)
plt.imshow(valor,cmap="viridis"); plt.colorbar(label="max Q"); plt.title("Función de valor aprendida"); plt.show()

estado=INICIO; ruta=[estado]
for _ in range(50):
    estado,_,fin=paso(estado,np.argmax(Q[índice(estado)])); ruta.append(estado)
    if fin: break
print("ruta:",ruta,"pasos:",len(ruta)-1)


La recompensa define lo que el agente optimiza, no necesariamente lo que
queríamos. Este entorno conoce todos sus estados; redes profundas se
vuelven útiles cuando el estado es grande o continuo.

**Ejercicios:** quite descuento; haga obstáculos estocásticos; compare
SARSA; diseñe una recompensa que produzca una conducta indeseada; reporte
media e intervalo de retornos en 20 semillas, no sólo la mejor corrida.
